# V5 Training — FIXED (games ×3 + 6M puzzles + draws + resigns)

**Two critical fixes:**
1. **Cell 5** auto-fetches the Witty shards from `GM-games-with-eval` → `chess-data` (the previous notebook tried to add them but they weren't on `chess-data` yet).
2. **Cell 6** rebuilds `index.json` from `sync.list_data()` — the *actual* files on HF — not from the old index. Previous version emptied the index because it read only puzzle entries.

**Setup:**
- Accelerator: **GPU T4 ×2**
- Secret: **HF_TOKEN**
- Dataset: **omarcodeplay/gm-dataset**

In [24]:
# ═══ Cell 1 — Bootstrap ═══
import os, shutil, pathlib, sys

SRC_IN = '/kaggle/input/datasets/omarcodeplay/gm-dataset/src'
SRC_OUT = '/kaggle/working/src'
os.makedirs(SRC_OUT, exist_ok=True)

if not os.path.isdir(SRC_IN):
    import glob
    candidates = glob.glob('/kaggle/input/**/src', recursive=True)
    if candidates:
        SRC_IN = candidates[0]
    else:
        raise SystemExit(f'Source not found. Add omarcodeplay/gm-dataset.')

copied = []
for f in pathlib.Path(SRC_IN).glob('*.py'):
    shutil.copy(f, pathlib.Path(SRC_OUT) / f.name)
    copied.append(f.name)
print(f'✅ Copied {len(copied)} files: {sorted(copied)}')

✅ Copied 11 files: ['bench.py', 'build_data.py', 'download_data.py', 'encode.py', 'engine.py', 'gauntlet.py', 'model.py', 'selftest.py', 'sign_check.py', 'sync.py', 'train.py']


In [25]:
# ═══ Cell 2 — Install deps ═══
!pip install -q --upgrade python-chess huggingface_hub pandas numpy orjson zstandard bottleneck

In [26]:
# ═══ Cell 3 — Patch sync.py + engine.py + load sync ═══
import os, sys, pathlib
from kaggle_secrets import UserSecretsClient
os.environ['HF_TOKEN'] = UserSecretsClient().get_secret('HF_TOKEN')
os.environ['HF_USER'] = 'OmarCodePlay'

# Rewrite sync.py with env precedence
sp = pathlib.Path('/kaggle/working/src/sync.py')
sp.write_text('''import os
from huggingface_hub import HfApi, hf_hub_download

TOKEN = os.environ.get("HF_TOKEN")
if not TOKEN:
    try:
        from kaggle_secrets import UserSecretsClient
        TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
    except Exception:
        TOKEN = None
if not TOKEN:
    raise RuntimeError("HF_TOKEN not found")

USER = os.environ.get("HF_USER", "YOUR_HF_USERNAME")
MODEL_REPO = f"{USER}/chess-ckpt"
DATA_REPO = f"{USER}/chess-data"
api = HfApi(token=TOKEN)

def init():
    api.create_repo(MODEL_REPO, private=True, exist_ok=True)
    api.create_repo(DATA_REPO, private=True, repo_type="dataset", exist_ok=True)

def push_ckpt(path, name=None):
    api.upload_file(path_or_fileobj=path, path_in_repo=name or os.path.basename(path), repo_id=MODEL_REPO)

def pull_ckpt(name, d, force=False):
    local = os.path.join(d, name)
    if not force and os.path.exists(local):
        return local
    try:
        return hf_hub_download(MODEL_REPO, name, local_dir=d, token=TOKEN, force_download=force)
    except Exception:
        return None

def push_data(path, name=None):
    api.upload_file(path_or_fileobj=path, path_in_repo=name or os.path.basename(path), repo_id=DATA_REPO, repo_type="dataset")

def pull_data(name, d):
    return hf_hub_download(DATA_REPO, name, repo_type="dataset", local_dir=d, token=TOKEN)

def list_data():
    return [f.rfilename for f in api.list_repo_tree(DATA_REPO, repo_type="dataset")]
''')
print('✅ sync.py rewritten')

# Patch engine.py zobrist bug
ep = pathlib.Path('/kaggle/working/src/engine.py')
if ep.exists():
    esrc = ep.read_text()
    if 'board.zobrist_hash()' in esrc:
        esrc = esrc.replace('board.zobrist_hash()', 'chess.polyglot.zobrist_hash(board)')
        if 'import chess.polyglot' not in esrc:
            esrc = esrc.replace('import chess.syzygy', 'import chess.syzygy\nimport chess.polyglot')
        ep.write_text(esrc)
        print('✅ engine.py zobrist fixed')

# Load sync
for k in list(sys.modules):
    if k == 'sync' or k.startswith('sync.'):
        del sys.modules[k]
sys.path.insert(0, '/kaggle/working/src')
import sync
print(f'✅ sync loaded, token={sync.TOKEN[:6]}...')

✅ sync.py rewritten
✅ engine.py zobrist fixed
✅ sync loaded, token=hf_RlB...


In [27]:
# ═══ Cell 4 — Inspect chess-data + fail loudly if base shards missing ═══
existing = sync.list_data()
print(f'=== chess-data: {len(existing)} files ===')
for f in sorted(existing):
    print(f'  {f}')

# Base shards that MUST exist for training
critical = ['train_0000.npz', 'train_0001.npz', 'val.npz', 'test.npz']
missing_critical = [c for c in critical if c not in existing]

if missing_critical:
    print(f'\n❌ FATAL: missing base shards from chess-data:')
    for c in missing_critical:
        print(f'   {c}')
    print()
    print('   These are the game shards + val/test set built by build_data.py.')
    print('   Upload them from your local PC to chess-data:')
    print('   https://huggingface.co/datasets/OmarCodePlay/chess-data/upload/main')
    raise SystemExit('Cannot continue — base shards missing')

print(f'\n✅ All base shards present: {critical}')

# What's missing from the witty set
witty_need = ['draws_witty_0000.npz', 'resign_witty_0000.npz']
missing_witty = [w for w in witty_need if w not in existing]
print(f'\nWitty shards to add: {missing_witty if missing_witty else "none — all present"}')

=== chess-data: 23 files ===
  .gitattributes
  index.json
  puzz_0000.npz
  puzz_0001.npz
  puzz_0002.npz
  puzz_0003.npz
  puzzles_0000.npz
  puzzles_0001.npz
  puzzles_0002.npz
  puzzles_0003.npz
  puzzles_0004.npz
  puzzles_0005.npz
  puzzles_0006.npz
  puzzles_0007.npz
  puzzles_0008.npz
  puzzles_0009.npz
  puzzles_0010.npz
  puzzles_0011.npz
  puzzles_0012.npz
  test.npz
  train_0000.npz
  train_0001.npz
  val.npz

✅ All base shards present: ['train_0000.npz', 'train_0001.npz', 'val.npz', 'test.npz']

Witty shards to add: ['draws_witty_0000.npz', 'resign_witty_0000.npz']


In [30]:
# ═══ Cell 5 — Auto-fetch missing shards from GM-games-with-eval -> chess-data ═══
import os, shutil
from huggingface_hub import hf_hub_download

ARCHIVE_REPO = 'OmarCodePlay/GM-games-with-eval'
os.makedirs('/kaggle/working/fetched', exist_ok=True)

# Shards we want in chess-data, source repo lookup order
# (draws_0000/resign_0000 are Hikaru; draws_witty/resign_witty are Witty_Alien)
to_fetch = [
    'draws_0000.npz',
    'resign_0000.npz',
    'draws_witty_0000.npz',
    'Witty_Alien_resigns_26.csv',
    'Hikaru-Moves-2024.csv',
    'Witty_Alien_resigns.csv',
]

existing_now = set(sync.list_data())
for name in to_fetch:
    if name in existing_now:
        print(f'✅ {name} already in chess-data')
        continue

    # Try to pull from archive repo
    try:
        print(f'⬇️  fetching {name} from {ARCHIVE_REPO}...')
        local = hf_hub_download(
            ARCHIVE_REPO, name,
            repo_type='dataset',
            local_dir='/kaggle/working/fetched',
            token=os.environ['HF_TOKEN'])
        size_mb = os.path.getsize(local) / 1e6
        print(f'   ✅ downloaded ({size_mb:.2f} MB)')

        # Upload to chess-data
        print(f'⬆️  pushing {name} to chess-data...')
        sync.push_data(local, name)
        print(f'   ✅ uploaded')
    except Exception as e:
        print(f'   ⚠️  could not fetch {name}: {e}')
        print(f'      → upload manually to chess-data from PC if you have it locally')

# Final state
final_files = sync.list_data()
print(f'\n=== chess-data after fetch: {len(final_files)} files ===')
for name in to_fetch:
    print(f"{'✅' if name in final_files else '❌'} {name}")

✅ draws_0000.npz already in chess-data
✅ resign_0000.npz already in chess-data
✅ draws_witty_0000.npz already in chess-data
⬇️  fetching Witty_Alien_resigns_26.csv from OmarCodePlay/GM-games-with-eval...


Witty_Alien_resigns_26.csv:   0%|          | 0.00/664k [00:00<?, ?B/s]

   ✅ downloaded (0.66 MB)
⬆️  pushing Witty_Alien_resigns_26.csv to chess-data...
   ✅ uploaded
✅ Hikaru-Moves-2024.csv already in chess-data
⬇️  fetching Witty_Alien_resigns.csv from OmarCodePlay/GM-games-with-eval...


Witty_Alien_resigns.csv:   0%|          | 0.00/610k [00:00<?, ?B/s]

   ✅ downloaded (0.61 MB)
⬆️  pushing Witty_Alien_resigns.csv to chess-data...
   ✅ uploaded

=== chess-data after fetch: 29 files ===
✅ draws_0000.npz
✅ resign_0000.npz
✅ draws_witty_0000.npz
✅ Witty_Alien_resigns_26.csv
✅ Hikaru-Moves-2024.csv
✅ Witty_Alien_resigns.csv


In [35]:
# ═══ Cell 6 — REBUILD index.json (all 4 value shards) ═══
import json

files = set(sync.list_data())
print(f'=== chess-data has {len(files)} files ===\n')

# ── Step 1: discover game shards ──
game_shards = sorted([f for f in files
                      if f.startswith('train_') and f.endswith('.npz')])
print(f'Game shards: {len(game_shards)}  →  {game_shards}\n')

if not game_shards:
    raise SystemExit('❌ FATAL: no train_*.npz on chess-data')

# ── Step 2: validate val/test ──
for req in ['val.npz', 'test.npz']:
    if req not in files:
        raise SystemExit(f'❌ FATAL: {req} missing')
print(f'✅ val.npz and test.npz present\n')

# ── Step 3: build train list ──
train = []

# Games × 3
for _ in range(3):
    train.extend(game_shards)
n_games_entries = len(train)

# All puzzle shards
puzzles = sorted([f for f in files if f.startswith('puzzles_') and f.endswith('.npz')])
train.extend(puzzles)

# ALL value shards (4 total)
value_shards = []
for name in ['draws_0000.npz',
             'resign_0000.npz',
             'draws_witty_0000.npz',
             'resign_witty_26_0000.npz']:   # ← NEW 2026 shard
    if name in files:
        train.append(name)
        value_shards.append(name)

if not puzzles:
    raise SystemExit('❌ FATAL: no puzzle shards')

# ── Step 4: write + upload ──
idx = {'train': train, 'val': ['val.npz'], 'test': ['test.npz']}
with open('/tmp/index.json', 'w') as f:
    json.dump(idx, f, indent=2)
sync.push_data('/tmp/index.json', 'index.json')

# ── Step 5: report ──
print(f'✅ index.json rebuilt with {len(train)} train entries\n')
print(f'  Games (×3):   {n_games_entries} entries  ({len(game_shards)} shards × 3)')
print(f'  Puzzles:      {len(puzzles)} entries')
print(f'  Value shards: {len(value_shards)} entries\n')

print('Full train list:')
for s in train:
    print(f'  {s}')
print()

# ── Step 6: value shard verification ──
print('Value shard verification:')
for name in ['draws_0000.npz',
             'resign_0000.npz',
             'draws_witty_0000.npz',
             'resign_witty_26_0000.npz']:
    if name in train:
        print(f'  ✅ {name}')
    else:
        print(f'  ❌ {name} — MISSING from index')
print()

# ── Step 7: mix estimate ──
samples_per_game_shard = 809_000 // max(len(game_shards), 1)
n_games = samples_per_game_shard * len(game_shards) * 3
n_puzzles = len(puzzles) * 475_000

# Real per-shard sizes
value_samples = {
    'draws_0000.npz': 60_541,
    'resign_0000.npz': 5_411,
    'draws_witty_0000.npz': 53_172,
    'resign_witty_26_0000.npz': 4_160,
}
n_value = sum(value_samples.get(n, 0) for n in value_shards)
total = n_games + n_puzzles + n_value

print(f'Estimated effective mix (per epoch):')
print(f'  Games:   ~{n_games:,}  ({100*n_games/total:.1f}%)')
print(f'  Puzzles: ~{n_puzzles:,}  ({100*n_puzzles/total:.1f}%)')
print(f'  Value:   ~{n_value:,}  ({100*n_value/total:.1f}%)')
print(f'  ──────────────────────')
print(f'  Total:   ~{total:,}')

# ── Step 8: max_steps ──
epochs = 6
steps = int(epochs * total / 2048)
print(f'\nRecommended max_steps={steps:,} (~{epochs} epochs)')
print(f'Estimated time: ~{steps * 2048 / 17000 / 60:.0f} min')

if n_games > 0 and n_puzzles > 0 and len(value_shards) == 4:
    print('\n✅ Mix is healthy — safe to train')
elif n_games > 0 and n_puzzles > 0 and len(value_shards) >= 2:
    print(f'\n⚠️  Only {len(value_shards)}/4 value shards — proceed but weaker value head')
else:
    print('\n❌ Mix has critical issues')

=== chess-data has 30 files ===

Game shards: 2  →  ['train_0000.npz', 'train_0001.npz']

✅ val.npz and test.npz present

✅ index.json rebuilt with 23 train entries

  Games (×3):   6 entries  (2 shards × 3)
  Puzzles:      13 entries
  Value shards: 4 entries

Full train list:
  train_0000.npz
  train_0001.npz
  train_0000.npz
  train_0001.npz
  train_0000.npz
  train_0001.npz
  puzzles_0000.npz
  puzzles_0001.npz
  puzzles_0002.npz
  puzzles_0003.npz
  puzzles_0004.npz
  puzzles_0005.npz
  puzzles_0006.npz
  puzzles_0007.npz
  puzzles_0008.npz
  puzzles_0009.npz
  puzzles_0010.npz
  puzzles_0011.npz
  puzzles_0012.npz
  draws_0000.npz
  resign_0000.npz
  draws_witty_0000.npz
  resign_witty_26_0000.npz

Value shard verification:
  ✅ draws_0000.npz
  ✅ resign_0000.npz
  ✅ draws_witty_0000.npz
  ✅ resign_witty_26_0000.npz

Estimated effective mix (per epoch):
  Games:   ~2,427,000  (27.8%)
  Puzzles: ~6,175,000  (70.8%)
  Value:   ~123,284  (1.4%)
  ──────────────────────
  Total:   ~8,

In [36]:
# ═══ Cell 7 — Patch train.py (v_mask + pol_loss + v5 config) ═══
import pathlib
p = pathlib.Path('/kaggle/working/src/train.py')
src = p.read_text()

src = src.replace('map_location=local_rank', 'map_location=f"cuda:{local_rank}"')

# PATCH 1: load_chunk — add v_mask
old_load = '''def load_chunk(names, device):
    arr = {k: [] for k in KEYS}
    local = os.path.join(DATA, f"r{device}")
    os.makedirs(local, exist_ok=True)
    for n in names:
        p = os.path.join(local, n)
        if not os.path.exists(p):
            p = sync.pull_data(n, local)
        z = np.load(p)
        for k in KEYS:
            arr[k].append(z[k])
    return {k: torch.from_numpy(np.concatenate(v)).to(device) for k, v in arr.items()}'''

new_load = '''def load_chunk(names, device):
    arr = {k: [] for k in KEYS}
    vmasks = []
    local = os.path.join(DATA, f"r{device}")
    os.makedirs(local, exist_ok=True)
    for n in names:
        p = os.path.join(local, n)
        if not os.path.exists(p):
            p = sync.pull_data(n, local)
        z = np.load(p)
        for k in KEYS:
            arr[k].append(z[k])
        is_puzzle = n.startswith("puzzles_")
        vm = np.zeros(z["val"].shape[0], dtype=np.float16)
        if not is_puzzle:
            vm.fill(1.0)
        vmasks.append(vm)
    out = {k: torch.from_numpy(np.concatenate(v)).to(device) for k, v in arr.items()}
    out["vmask"] = torch.from_numpy(np.concatenate(vmasks)).to(device)
    return out'''

if old_load in src:
    src = src.replace(old_load, new_load)
    print('✅ load_chunk patched')
elif 'vmask' in src:
    print('✅ load_chunk already patched')
else:
    print('❌ load_chunk NOT found')

# PATCH 2: training loop uses v_mask
old_loop = '''                b = perm_local[i: i + B]
                bb, meta, mvk, wk, v = [data[k][b] for k in KEYS]
                for g in opt.param_groups:
                    g["lr"] = lr_at(st["step"])
                with torch.autocast("cuda", dtype=torch.float16):
                    lg, pv = ddp(bb, meta)
                lp = pol_loss(lg, mvk, wk, C["smooth"])
                lv = F.mse_loss(pv.float(), v)
                loss = lp + C["v_w"] * lv'''

new_loop = '''                b = perm_local[i: i + B]
                bb, meta, mvk, wk, v = [data[k][b] for k in KEYS]
                vmask = data["vmask"][b].float()
                for g in opt.param_groups:
                    g["lr"] = lr_at(st["step"])
                with torch.autocast("cuda", dtype=torch.float16):
                    lg, pv = ddp(bb, meta)
                lp = pol_loss(lg, mvk, wk, C["smooth"])
                lv_raw = F.mse_loss(pv.float(), v, reduction="none")
                lv = (lv_raw * vmask).sum() / vmask.sum().clamp_min(1.0)
                loss = lp + C["v_w"] * lv'''

if old_loop in src:
    src = src.replace(old_loop, new_loop)
    print('✅ training loop patched')
elif 'vmask = data["vmask"]' in src:
    print('✅ training loop already patched')
else:
    print('❌ training loop NOT found')

# PATCH 3: pol_loss masking
old_pol = '''def pol_loss(lg, mvk, wk, smooth):
    lp = F.log_softmax(lg.float(), 1)
    w = wk.float()
    w = w / w.sum(1, keepdim=True).clamp_min(1e-6)
    nll = -(w * lp.gather(1, mvk.long().clamp_min(0))).sum(1).mean()
    return (1 - smooth) * nll + smooth * (-lp.mean(1)).mean()'''

new_pol = '''def pol_loss(lg, mvk, wk, smooth):
    has_policy = (wk.sum(dim=1) > 0)
    if not has_policy.any():
        return torch.zeros((), device=lg.device, dtype=torch.float32)
    lg_m = lg[has_policy]
    mvk_m = mvk[has_policy]
    wk_m = wk[has_policy]
    lp = F.log_softmax(lg_m.float(), 1)
    w = wk_m.float()
    w = w / w.sum(1, keepdim=True).clamp_min(1e-6)
    nll = -(w * lp.gather(1, mvk_m.long().clamp_min(0))).sum(1).mean()
    return (1 - smooth) * nll + smooth * (-lp.mean(1)).mean()'''

if old_pol in src:
    src = src.replace(old_pol, new_pol)
    print('✅ pol_loss masked')
elif 'has_policy' in src:
    print('✅ pol_loss already masked')
else:
    print('❌ pol_loss NOT found')

# PATCH 4: v5 config
for old_run in ['base2', 'base3', 'base4', 'base5', 'base6', 'base7', 'v5']:
    src = src.replace(f'run="{old_run}",', 'run="v5",')

if 'init_from="engine.pt",' not in src:
    for old_init in ['init_from=None,', 'init_from="best_base3.pt",', 'init_from="best_base2.pt",']:
        src = src.replace(old_init, 'init_from="engine.pt",')

for old in ['lr=2e-3,', 'lr=3e-5,', 'lr=4e-5,', 'lr=5e-5,']:
    src = src.replace(old, 'lr=2e-5,')
for old in ['v_w=2.0,', 'v_w=0.8,', 'v_w=0.7,', 'v_w=0.5,', 'v_w=0.3,']:
    src = src.replace(old, 'v_w=1.0,')
for old in ['max_steps=12000,', 'max_steps=14000,', 'max_steps=15000,', 'max_steps=10000,', 'max_steps=None,']:
    src = src.replace(old, 'max_steps=25562,')
for old in ['decay_start=9000,', 'decay_start=10000,', 'decay_start=11000,', 'decay_start=7500,', 'decay_start=None,']:
    src = src.replace(old, 'decay_start=20000,')
for old in ['decay_steps=4000,', 'decay_steps=3000,', 'decay_steps=2500,', 'decay_steps=100_000,']:
    src = src.replace(old, 'decay_steps=5500,')
if 'patience' not in src:
    src = src.replace('max_steps=25562,', 'max_steps=25562,\n    patience=3,\n    valmse_tolerance=0.0005,')

p.write_text(src)
print('\n✅ train.py configured for v5')
print('=' * 60)
!grep -nE '^    run=|^    init_from=|^    lr=|^    v_w=|^    max_steps=|^    patience=' /kaggle/working/src/train.py

✅ load_chunk patched
✅ training loop patched
✅ pol_loss already masked

✅ train.py configured for v5
25:    run="v5",
26:    init_from="engine.pt",
30:    lr=2e-5,
35:    max_steps=25562,
36:    v_w=1.0,
44:    patience=3,


In [37]:
# ═══ Cell 8 — Verify + launch training ═══
import os, sys, json, subprocess
from kaggle_secrets import UserSecretsClient
os.environ['HF_TOKEN'] = UserSecretsClient().get_secret('HF_TOKEN')
os.environ['HF_USER'] = 'OmarCodePlay'
for k in list(sys.modules):
    if k == 'sync' or k.startswith('sync.'):
        del sys.modules[k]
sys.path.insert(0, '/kaggle/working/src')
import sync

idx = json.load(open(sync.pull_data('index.json', '/kaggle/working/shards')))
print(f'Train shards ({len(idx["train"])}):')
for s in idx['train']:
    print(f'  {s}')
print(f'Val:  {idx["val"]}')
print(f'Test: {idx["test"]}\n')

checks_ok = True
if len(idx['train']) < 20:
    print(f'❌ Only {len(idx["train"])} entries')
    checks_ok = False

r = subprocess.run(['grep', '-c', 'vmask', '/kaggle/working/src/train.py'],
                   capture_output=True, text=True)
vmask_count = int(r.stdout.strip() or 0)
print(f'vmask occurrences: {vmask_count} (need ≥3)')
if vmask_count < 3:
    checks_ok = False

r = subprocess.run(['grep', '-c', 'has_policy', '/kaggle/working/src/train.py'],
                   capture_output=True, text=True)
policy_count = int(r.stdout.strip() or 0)
print(f'has_policy occurrences: {policy_count} (need ≥2)')
if policy_count < 2:
    checks_ok = False

import torch
ck = sync.pull_ckpt('engine.pt', '/kaggle/working/ckpt')
print(f"engine.pt: {'✅ ' + str(round(os.path.getsize(ck)/1e6, 1)) + ' MB' if ck else '❌ missing'}")
if not ck:
    checks_ok = False

print('=' * 60)
if checks_ok:
    print('✅ All checks passed — launching training\n')
    !cd /kaggle/working && torchrun --standalone --nproc_per_node=2 src/train.py 2>&1 | tee -a /kaggle/working/train_v5.log
else:
    print('❌ Some checks failed — not launching')

index.json:   0%|          | 0.00/630 [00:00<?, ?B/s]

Train shards (23):
  train_0000.npz
  train_0001.npz
  train_0000.npz
  train_0001.npz
  train_0000.npz
  train_0001.npz
  puzzles_0000.npz
  puzzles_0001.npz
  puzzles_0002.npz
  puzzles_0003.npz
  puzzles_0004.npz
  puzzles_0005.npz
  puzzles_0006.npz
  puzzles_0007.npz
  puzzles_0008.npz
  puzzles_0009.npz
  puzzles_0010.npz
  puzzles_0011.npz
  puzzles_0012.npz
  draws_0000.npz
  resign_0000.npz
  draws_witty_0000.npz
  resign_witty_26_0000.npz
Val:  ['val.npz']
Test: ['test.npz']

vmask occurrences: 5 (need ≥3)
has_policy occurrences: 5 (need ≥2)


engine.pt: reconstructing file:   0%|          |  0.00B / 46.4MB            

engine.pt: downloading bytes:           |  0.00B            

engine.pt: ✅ 46.4 MB
✅ All checks passed — launching training

W1111 23:12:27.300000 542 torch/distributed/run.py:851] 
W1111 23:12:27.300000 542 torch/distributed/run.py:851] *****************************************
W1111 23:12:27.300000 542 torch/distributed/run.py:851] Setting OMP_NUM_THREADS environment variable for each process to be 1 in default, to avoid your system being overloaded, please further tune the variable for optimal performance in your application as needed. 
W1111 23:12:27.300000 542 torch/distributed/run.py:851] *****************************************
/usr/local/lib/python3.13/dist-packages/torch/distributed/c10d_logger.py:83: UserWarning: barrier(): using the device under current context. You can specify `device_id` in `init_process_group` to mute this warning.
  return func(*args, **kwargs)
[rank0]:[W1111 23:12:30.841744230 ProcessGroupNCCL.cpp:5188] Guessing device ID based on global rank. This can cause a hang if rank to GPU mapping is heterogeneous. You can

In [ ]:
# ═══ Cell 9 — Post-training tests (run after SESSION END) ═══
import chess, pathlib, sys
sys.path.insert(0, '/kaggle/working/src')

ep = pathlib.Path('/kaggle/working/src/engine.py')
esrc = ep.read_text()
if 'board.zobrist_hash()' in esrc:
    esrc = esrc.replace('board.zobrist_hash()', 'chess.polyglot.zobrist_hash(board)')
    if 'import chess.polyglot' not in esrc:
        esrc = esrc.replace('import chess.syzygy', 'import chess.syzygy\nimport chess.polyglot')
    ep.write_text(esrc)

from engine import Engine

print('=' * 60)
print('TEST 1: Mate-in-1 (back rank) — v5 should play mate')
print('=' * 60)
fen = '6k1/5ppp/8/8/8/8/5PPP/R5K1 w - - 0 1'
board = chess.Board(fen)
v5 = Engine('/kaggle/working/ckpt/best_v5.pt')
v3 = Engine('/kaggle/working/ckpt/engine.pt')
mv5, _ = v5.search(board, float('inf'), float('inf'), 500, None)
mv3, _ = v3.search(board, float('inf'), float('inf'), 500, None)
bc = board.copy(); bc.push(mv5)
print(f'v5: {mv5.uci()} -> {"MATE ✅" if bc.is_checkmate() else "not mate ❌"}')
print(f'v3: {mv3.uci()}')

print('\n' + '=' * 60)
print('TEST 2: Drawn position — should be near 0')
print('=' * 60)
bd = chess.Board('8/8/4k3/8/8/4K3/8/R6r w - - 0 1')
_, r5 = v5.search(bd, float('inf'), float('inf'), 200, None)
_, r3 = v3.search(bd, float('inf'), float('inf'), 200, None)
val5 = r5.w / r5.n if r5 and r5.n else 0
val3 = r3.w / r3.n if r3 and r3.n else 0
print(f'v5: {val5:+.3f}  {"✅ near 0" if abs(val5) < 0.3 else "⚠️ overconfident"}')
print(f'v3: {val3:+.3f}  {"✅ near 0" if abs(val3) < 0.3 else "⚠️ overconfident"}')

print('\n' + '=' * 60)
print('TEST 3: Won position — should be near +1')
print('=' * 60)
bw = chess.Board('rnb1kbnr/pppp1ppp/8/8/8/8/PPPP1PPP/RNBQKBNR w KQkq - 0 1')
_, r5w = v5.search(bw, float('inf'), float('inf'), 200, None)
_, r3w = v3.search(bw, float('inf'), float('inf'), 200, None)
val5w = r5w.w / r5w.n if r5w and r5w.n else 0
val3w = r3w.w / r3w.n if r3w and r3w.n else 0
print(f'v5: {val5w:+.3f}  {"✅ near +1" if val5w > 0.5 else "⚠️ weak"}')
print(f'v3: {val3w:+.3f}  {"✅ near +1" if val3w > 0.5 else "⚠️ weak"}')

In [40]:
# ═══ Patch train.py for curriculum: trunk_lr_scale + trainable_prefixes ═══
import pathlib
p = pathlib.Path('/kaggle/working/src/train.py')
src = p.read_text()

# Patch 1: Add curriculum fields to C dict (if not already there)
if 'trunk_lr_scale' not in src:
    src = src.replace(
        'max_steps=25562,',
        'max_steps=25562,\n    trunk_lr_scale=1.0,\n    trainable_prefixes=None,'
    )
    print('✅ Added trunk_lr_scale + trainable_prefixes to C')

# Patch 2: After model init, apply trainable_prefixes freeze
old_init = 'raw = ChessNet(C["filters"], C["blocks"]).to(local_rank) \\\n        .to(memory_format=torch.channels_last)'
new_init = '''raw = ChessNet(C["filters"], C["blocks"]).to(local_rank) \\
        .to(memory_format=torch.channels_last)

    # ── Curriculum: optionally freeze by prefix ──
    if C.get("trainable_prefixes"):
        prefixes = tuple(p.strip() for p in C["trainable_prefixes"].split(",") if p.strip())
        n_frozen = 0
        n_trainable = 0
        for name, param in raw.named_parameters():
            if any(name.startswith(p) for p in prefixes):
                param.requires_grad = True
                n_trainable += 1
            else:
                param.requires_grad = False
                n_frozen += 1
        if is_main:
            print(f"[curriculum] trainable_prefixes={prefixes}", flush=True)
            print(f"[curriculum] {n_trainable} trainable, {n_frozen} frozen", flush=True)'''

if old_init in src and 'Curriculum: optionally freeze' not in src:
    src = src.replace(old_init, new_init)
    print('✅ Added trainable_prefixes freeze after model init')

# Patch 3: Replace optimizer with param groups (trunk vs heads)
old_opt = 'opt = torch.optim.AdamW(model.parameters(), lr=C["lr"], weight_decay=C["wd"])'
new_opt = '''# Curriculum: split trunk vs head param groups for lr_scale
    trunk_params = []
    head_params = []
    for name, param in model.named_parameters():
        if not param.requires_grad:
            continue
        if name.startswith(('stem.', 'body.')):
            trunk_params.append(param)
        else:
            head_params.append(param)
    if C.get("trunk_lr_scale", 1.0) != 1.0 and trunk_params:
        opt = torch.optim.AdamW([
            {"params": trunk_params, "lr": C["lr"] * C["trunk_lr_scale"]},
            {"params": head_params, "lr": C["lr"]},
        ], weight_decay=C["wd"])
        if is_main:
            print(f"[curriculum] trunk lr_scale={C['trunk_lr_scale']} "
                  f"({len(trunk_params)} tensors), {len(head_params)} head tensors", flush=True)
    else:
        opt = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad],
                                lr=C["lr"], weight_decay=C["wd"])'''

if old_opt in src and 'Curriculum: split trunk vs head' not in src:
    src = src.replace(old_opt, new_opt)
    print('✅ Replaced optimizer with param groups')

# Patch 4: lr_at scaling for param groups
old_lr_apply = '''                for g in opt.param_groups:
                    g["lr"] = lr_at(st["step"])'''
new_lr_apply = '''                base_lr = lr_at(st["step"])
                for gi, g in enumerate(opt.param_groups):
                    mult = C.get("trunk_lr_scale", 1.0) if (gi == 0 and len(opt.param_groups) == 2) else 1.0
                    g["lr"] = base_lr * mult'''

if old_lr_apply in src:
    src = src.replace(old_lr_apply, new_lr_apply)
    print('✅ Patched lr_at apply for param groups')

p.write_text(src)
print('\n✅ train.py patched for curriculum')
print('=' * 60)

# Verify
!grep -nE 'trunk_lr_scale|trainable_prefixes' /kaggle/working/src/train.py

✅ Added trunk_lr_scale + trainable_prefixes to C
✅ Added trainable_prefixes freeze after model init
✅ Replaced optimizer with param groups
✅ Patched lr_at apply for param groups

✅ train.py patched for curriculum
36:    trunk_lr_scale=1.0,
37:    trainable_prefixes=None,
166:    if C.get("trainable_prefixes"):
167:        prefixes = tuple(p.strip() for p in C["trainable_prefixes"].split(",") if p.strip())
178:            print(f"[curriculum] trainable_prefixes={prefixes}", flush=True)
192:    if C.get("trunk_lr_scale", 1.0) != 1.0 and trunk_params:
194:            {"params": trunk_params, "lr": C["lr"] * C["trunk_lr_scale"]},
198:            print(f"[curriculum] trunk lr_scale={C['trunk_lr_scale']} "
285:                    mult = C.get("trunk_lr_scale", 1.0) if (gi == 0 and len(opt.param_groups) == 2) else 1.0


In [41]:
# ═══ STAGE A — build index_A.json (puzzles only) ═══
import json
files = set(sync.list_data())
puzzles = sorted([f for f in files if f.startswith('puzzles_') and f.endswith('.npz')])

idx_A = {'train': puzzles, 'val': ['val.npz'], 'test': ['test.npz']}
with open('/tmp/index_A.json', 'w') as f:
    json.dump(idx_A, f)
sync.push_data('/tmp/index_A.json', 'index.json')
print(f'✅ index_A.json: {len(puzzles)} puzzle shards only')

# ═══ STAGE A — configure train.py ═══
import pathlib
p = pathlib.Path('/kaggle/working/src/train.py')
src = p.read_text()

# Stage A config
src = src.replace('run="v5",', 'run="v6a",')
src = src.replace('init_from="engine.pt",', 'init_from="engine.pt",')  # unchanged
src = src.replace('lr=2e-5,', 'lr=5e-5,')                               # heads can be aggressive
src = src.replace('v_w=1.0,', 'v_w=0.0,')                               # NO value training on puzzles
src = src.replace('max_steps=25562,', 'max_steps=3000,')
src = src.replace('decay_start=20000,', 'decay_start=2000,')
src = src.replace('decay_steps=5500,', 'decay_steps=1000,')
src = src.replace('patience=3,', 'patience=0,')                          # no early stop
src = src.replace('trainable_prefixes=None,', 'trainable_prefixes="p_conv,p_fc",')
src = src.replace('trunk_lr_scale=1.0,', 'trunk_lr_scale=1.0,')

p.write_text(src)
print('✅ Stage A config: 3000 steps, puzzles only, trunk frozen, v_w=0')

!grep -nE '^    run=|^    lr=|^    v_w=|^    max_steps=|^    trainable_prefixes=' /kaggle/working/src/train.py

KeyboardInterrupt: 

In [42]:
import pathlib
p = pathlib.Path('/kaggle/working/src/train.py')
src = p.read_text()

# Add curriculum fields
if 'trunk_lr_scale' not in src:
    src = src.replace(
        'max_steps=25562,',
        'max_steps=25562,\n    trunk_lr_scale=1.0,\n    trainable_prefixes=None,'
    )

# Freeze-by-prefix
old_init = 'raw = ChessNet(C["filters"], C["blocks"]).to(local_rank) \\\n        .to(memory_format=torch.channels_last)'
new_init = '''raw = ChessNet(C["filters"], C["blocks"]).to(local_rank) \\
        .to(memory_format=torch.channels_last)

    if C.get("trainable_prefixes"):
        prefixes = tuple(p.strip() for p in C["trainable_prefixes"].split(",") if p.strip())
        n_frozen = n_trainable = 0
        for name, param in raw.named_parameters():
            if any(name.startswith(p) for p in prefixes):
                param.requires_grad = True
                n_trainable += 1
            else:
                param.requires_grad = False
                n_frozen += 1
        if is_main:
            print(f"[curriculum] trainable={prefixes} ({n_trainable} on, {n_frozen} off)", flush=True)'''

if old_init in src and 'Curriculum: freeze' not in src:
    src = src.replace(old_init, new_init)

# Param groups
old_opt = 'opt = torch.optim.AdamW(model.parameters(), lr=C["lr"], weight_decay=C["wd"])'
new_opt = '''trunk_params, head_params = [], []
    for name, param in model.named_parameters():
        if not param.requires_grad: continue
        if name.startswith(('stem.', 'body.')):
            trunk_params.append(param)
        else:
            head_params.append(param)
    if C.get("trunk_lr_scale", 1.0) != 1.0 and trunk_params:
        opt = torch.optim.AdamW([
            {"params": trunk_params, "lr": C["lr"] * C["trunk_lr_scale"]},
            {"params": head_params, "lr": C["lr"]},
        ], weight_decay=C["wd"])
        if is_main:
            print(f"[curriculum] trunk lr_scale={C['trunk_lr_scale']}", flush=True)
    else:
        opt = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad],
                                lr=C["lr"], weight_decay=C["wd"])'''
if old_opt in src and 'Curriculum: param groups' not in src:
    src = src.replace(old_opt, new_opt)

# LR apply with scale
old_lr = '''                for g in opt.param_groups:
                    g["lr"] = lr_at(st["step"])'''
new_lr = '''                base_lr = lr_at(st["step"])
                for gi, g in enumerate(opt.param_groups):
                    mult = C.get("trunk_lr_scale", 1.0) if (gi == 0 and len(opt.param_groups) == 2) else 1.0
                    g["lr"] = base_lr * mult'''
if old_lr in src:
    src = src.replace(old_lr, new_lr)

p.write_text(src)
print('✅ train.py patched for curriculum')
!grep -nE 'trunk_lr_scale|trainable_prefixes' /kaggle/working/src/train.py

✅ train.py patched for curriculum
36:    trunk_lr_scale=1.0,
37:    trainable_prefixes=None,
165:    if C.get("trainable_prefixes"):
166:        prefixes = tuple(p.strip() for p in C["trainable_prefixes"].split(",") if p.strip())
179:    if C.get("trainable_prefixes"):
180:        prefixes = tuple(p.strip() for p in C["trainable_prefixes"].split(",") if p.strip())
191:            print(f"[curriculum] trainable_prefixes={prefixes}", flush=True)
205:    if C.get("trunk_lr_scale", 1.0) != 1.0 and trunk_params:
207:            {"params": trunk_params, "lr": C["lr"] * C["trunk_lr_scale"]},
211:            print(f"[curriculum] trunk lr_scale={C['trunk_lr_scale']} "
298:                    mult = C.get("trunk_lr_scale", 1.0) if (gi == 0 and len(opt.param_groups) == 2) else 1.0


In [44]:
import os, sys
from kaggle_secrets import UserSecretsClient
os.environ['HF_TOKEN'] = UserSecretsClient().get_secret('HF_TOKEN')
os.environ['HF_USER'] = 'OmarCodePlay'
for k in list(sys.modules):
    if k == 'sync' or k.startswith('sync.'):
        del sys.modules[k]

!cd /kaggle/working && torchrun --standalone --nproc_per_node=2 src/train.py 2>&1 | tee -a /kaggle/working/train_v6a.log

W1111 23:45:27.259000 940 torch/distributed/run.py:851] 
W1111 23:45:27.259000 940 torch/distributed/run.py:851] *****************************************
W1111 23:45:27.259000 940 torch/distributed/run.py:851] Setting OMP_NUM_THREADS environment variable for each process to be 1 in default, to avoid your system being overloaded, please further tune the variable for optimal performance in your application as needed. 
W1111 23:45:27.259000 940 torch/distributed/run.py:851] *****************************************
/usr/local/lib/python3.13/dist-packages/torch/distributed/c10d_logger.py:83: UserWarning: barrier(): using the device under current context. You can specify `device_id` in `init_process_group` to mute this warning.
  return func(*args, **kwargs)
[rank0]:[W1111 23:45:30.575617581 ProcessGroupNCCL.cpp:5188] Guessing device ID based on global rank. This can cause a hang if rank to GPU mapping is heterogeneous. You can specify device_id in init_process_group()
CONFIG {'run': 'v6a'

In [63]:
import os, sys, time, importlib

# Clean environment
os.environ.pop("CHESS_CKPT", None)  # let each gauntlet subprocess set its own

sys.path.insert(0, '/kaggle/working')

# Reload gauntlet fresh (in case it's cached)
if 'gauntlet' in sys.modules:
    del sys.modules['gauntlet']
import gauntlet
importlib.reload(gauntlet)

# Verify files exist first
import pathlib
for f in ['/kaggle/working/ckpt/engine.pt',
          '/kaggle/working/ckpt/ema_v2.bias.pt']:
    p = pathlib.Path(f)
    print(f"  {'✅' if p.exists() else '❌'} {f}  "
          f"({p.stat().st_size/1e6:.1f} MB)" if p.exists() else f"  ❌ {f}")

print()
print("=" * 60)
print("SHORT GAUNTLET: 6 games @ 400 nodes (~4 min)")
print("A = engine.pt (base3)  |  B = ema_v2.bias.pt (v6b)")
print("=" * 60, flush=True)
print()

t0 = time.time()

# 3 pairs = 6 games (each pair = both colors)
gauntlet.ab(
    '/kaggle/working/ckpt/engine.pt',
    '/kaggle/working/ckpt/ema_v2.bias.pt',
    nodes=500,
    pairs=5,
)

elapsed = time.time() - t0
print(f"\nTotal time: {elapsed/60:.1f} min")

  ✅ /kaggle/working/ckpt/engine.pt  (46.4 MB)
  ✅ /kaggle/working/ckpt/ema_v2.bias.pt  (46.4 MB)

SHORT GAUNTLET: 6 games @ 400 nodes (~4 min)
A = engine.pt (base3)  |  B = ema_v2.bias.pt (v6b)

A vs B: 3.0/10  Elo diff ≈ -147 (±70)

Total time: 4.2 min


In [81]:
# ═══ Build game-only shards from the 4 new CSVs ═══
import os, sys, math
import pandas as pd, numpy as np, chess
sys.path.insert(0, '/kaggle/working/src')
from encode import pack_board, move_index, save_npz
import sync

def cp_to_v(cp):
    return 2.0 / (1.0 + math.exp(-0.00368208 * cp)) - 1.0

def parse_mate(x):
    if pd.isna(x): return None
    try: f = float(x)
    except: return None
    if f == 0: return None
    return 1.0 if f > 0 else -1.0

def build_shards(csv_path, start_idx, min_depth=12, blunder_cp=200, shard_size=400_000):
    print(f'\n─── {os.path.basename(csv_path)} ───')
    df = pd.read_csv(csv_path, low_memory=False)
    print(f'  raw rows: {len(df):,}')

    if 'eval_depth_before' in df.columns:
        df['eval_depth_before'] = pd.to_numeric(df['eval_depth_before'], errors='coerce')
        df = df[df['eval_depth_before'] >= min_depth]
        print(f'  after depth≥{min_depth}: {len(df):,}')

    if 'eval_cp_before' in df.columns and 'eval_cp_after' in df.columns and 'color_to_move' in df.columns:
        df['eval_cp_before'] = pd.to_numeric(df['eval_cp_before'], errors='coerce')
        df['eval_cp_after'] = pd.to_numeric(df['eval_cp_after'], errors='coerce')
        ok = df['eval_cp_before'].notna() & df['eval_cp_after'].notna()
        sign = np.where(df['color_to_move'].str.lower().str[0] == 'w', 1, -1)
        loss = (df['eval_cp_before'] - df['eval_cp_after']) * sign
        df = df[~(ok & (loss > blunder_cp))]
        print(f'  after blunder filter: {len(df):,}')

    if 'game_id' in df.columns and 'ply' in df.columns:
        df = df.drop_duplicates(subset=['game_id', 'ply'], keep='first')
        print(f'  after dedup: {len(df):,}')

    samples = []
    seen_fens = set()
    skipped = 0
    for i, (_, r) in enumerate(df.iterrows()):
        if i % 100000 == 0 and i > 0:
            print(f'    built {len(samples):,}/{i:,}', flush=True)
        try:
            fen = r['fen']
            if fen in seen_fens: continue
            seen_fens.add(fen)
            b = chess.Board(fen)
            m = chess.Move.from_uci(str(r['move_uci']))
            if m not in b.legal_moves:
                skipped += 1; continue
            black = b.turn == chess.BLACK

            mate_v = parse_mate(r.get('eval_mate_before'))
            if mate_v is not None:
                v = mate_v
            else:
                cp = r.get('eval_cp_before')
                if pd.isna(cp):
                    skipped += 1; continue
                v = cp_to_v(float(cp))
            if black: v = -v

            bb, meta = pack_board(b)
            mvk = [move_index(m, black), -1, -1, -1]
            wk = [1.0, 0.0, 0.0, 0.0]

            if pd.notna(r.get('best_move_uci')):
                try:
                    bm = chess.Move.from_uci(str(r['best_move_uci']))
                    if bm in b.legal_moves and bm != m:
                        mvk[1] = move_index(bm, black)
                        wk = [0.7, 0.3, 0.0, 0.0]
                except: pass

            samples.append((bb, meta,
                            np.array(mvk, dtype=np.int16),
                            np.array(wk, dtype=np.float16),
                            float(v)))
        except:
            skipped += 1; continue

    print(f'  → samples: {len(samples):,}  (skipped {skipped:,})')

    shard_names = []
    for j in range(0, len(samples), shard_size):
        chunk = samples[j:j+shard_size]
        idx = start_idx + j // shard_size
        fn = f'train_{idx:04d}.npz'
        path = f'/kaggle/working/shards/{fn}'
        save_npz(chunk, path)
        shard_names.append(fn)
        print(f'  saved {fn} ({len(chunk):,} rows, {os.path.getsize(path)/1e6:.1f} MB)')
    return shard_names


os.makedirs('/kaggle/working/shards', exist_ok=True)

all_new_shards = []
start_idx = 2  # train_0000 and train_0001 already exist

for csv_name in ['Hikaru-Moves-2026.csv', 'moves_2024.csv', 'moves_2025.csv', 'moves_2026.csv']:
    path = f'/kaggle/working/data/{csv_name}'
    if not os.path.exists(path):
        print(f'⚠️  {path} missing, skipping')
        continue
    new_shards = build_shards(path, start_idx)
    all_new_shards.extend(new_shards)
    start_idx += len(new_shards)

print(f'\n✅ Built {len(all_new_shards)} new shards:')
for s in all_new_shards:
    print(f'  {s}')


─── Hikaru-Moves-2026.csv ───
  raw rows: 422,824
  after depth≥12: 374,159
  after blunder filter: 357,950
  after dedup: 357,950
    built 98,338/100,000
    built 194,956/200,000
    built 290,609/300,000
  → samples: 345,674  (skipped 0)
  saved train_0002.npz (345,674 rows, 40.8 MB)

─── moves_2024.csv ───
  raw rows: 177,298
  after depth≥12: 157,061
  after blunder filter: 151,408
  after dedup: 151,408
    built 98,065/100,000
  → samples: 147,922  (skipped 0)
  saved train_0003.npz (147,922 rows, 17.5 MB)

─── moves_2025.csv ───
  raw rows: 197,581
  after depth≥12: 173,827
  after blunder filter: 167,228
  after dedup: 167,228
    built 98,407/100,000
  → samples: 163,689  (skipped 0)
  saved train_0004.npz (163,689 rows, 19.3 MB)

─── moves_2026.csv ───
  raw rows: 102,403
  after depth≥12: 90,409
  after blunder filter: 86,986
  after dedup: 86,986
  → samples: 85,430  (skipped 0)
  saved train_0005.npz (85,430 rows, 10.1 MB)

✅ Built 4 new shards:
  train_0002.npz
  train

In [82]:
import os, sys, json
from kaggle_secrets import UserSecretsClient
os.environ['HF_TOKEN'] = UserSecretsClient().get_secret('HF_TOKEN')
os.environ['HF_USER'] = 'OmarCodePlay'
for k in list(sys.modules):
    if k == 'sync' or k.startswith('sync.'):
        del sys.modules[k]
sys.path.insert(0, '/kaggle/working/src')
import sync

# Upload new shards
for fn in all_new_shards:
    sync.push_data(f'/kaggle/working/shards/{fn}', fn)
    print(f'✅ {fn}')

# Rebuild index
files = set(sync.list_data())
games = sorted([f for f in files if f.startswith('train_') and f.endswith('.npz')])
puzzles = sorted([f for f in files if f.startswith('puzzles_') and f.endswith('.npz')])
value = [n for n in ['draws_0000.npz', 'resign_0000.npz',
                     'draws_witty_0000.npz', 'resign_witty_26_0000.npz'] if n in files]

print(f'\nGames: {len(games)} shards')
print(f'Puzzles: {len(puzzles)} shards')
print(f'Value: {len(value)} shards')

# Build train list: games ×2, puzzles ×1, value ×1
train = []
for _ in range(2):
    train.extend(games)
train.extend(puzzles)
train.extend(value)

idx = {'train': train, 'val': ['val.npz'], 'test': ['test.npz']}
with open('/tmp/index.json', 'w') as f:
    json.dump(idx, f)
sync.push_data('/tmp/index.json', 'index.json')

n_gm = len(games) * 2 * 400_000
n_puzzles = len(puzzles) * 475_000
n_value = 123_000
total = n_gm + n_puzzles + n_value

print(f'\n✅ index.json rebuilt — {len(train)} entries')
print(f'  Games (×2):  {len(games)*2:3d} entries  ~{n_gm:,}  ({100*n_gm/total:.1f}%)')
print(f'  Puzzles:     {len(puzzles):3d} entries  ~{n_puzzles:,}  ({100*n_puzzles/total:.1f}%)')
print(f'  Value:       {len(value):3d} entries  ~{n_value:,}  ({100*n_value/total:.1f}%)')
print(f'  ─────────────────────────────────')
print(f'  Total:       ~{total:,} samples')

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Validating:           |   0%            

✅ train_0002.npz


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

... (6 outputs left out)

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Validating:           |   0%            

✅ train_0005.npz

Games: 6 shards
Puzzles: 13 shards
Value: 4 shards

✅ index.json rebuilt — 29 entries
  Games (×2):   12 entries  ~4,800,000  (43.3%)
  Puzzles:      13 entries  ~6,175,000  (55.6%)
  Value:         4 entries  ~123,000  (1.1%)
  ─────────────────────────────────
  Total:       ~11,098,000 samples


In [ ]:
import os, sys
from kaggle_secrets import UserSecretsClient
os.environ['HF_TOKEN'] = UserSecretsClient().get_secret('HF_TOKEN')
os.environ['HF_USER'] = 'OmarCodePlay'
for k in list(sys.modules):
    if k == 'sync' or k.startswith('sync.'):
        del sys.modules[k]

!cd /kaggle/working && torchrun --standalone --nproc_per_node=2 src/train.py 2>&1 | tee -a /kaggle/working/train_v8.log

W1112 01:09:09.751000 1426 torch/distributed/run.py:851] 
W1112 01:09:09.751000 1426 torch/distributed/run.py:851] *****************************************
W1112 01:09:09.751000 1426 torch/distributed/run.py:851] Setting OMP_NUM_THREADS environment variable for each process to be 1 in default, to avoid your system being overloaded, please further tune the variable for optimal performance in your application as needed. 
W1112 01:09:09.751000 1426 torch/distributed/run.py:851] *****************************************
/usr/local/lib/python3.13/dist-packages/torch/distributed/c10d_logger.py:83: UserWarning: barrier(): using the device under current context. You can specify `device_id` in `init_process_group` to mute this warning.
  return func(*args, **kwargs)
[rank0]:[W1112 01:09:12.998250069 ProcessGroupNCCL.cpp:5188] Guessing device ID based on global rank. This can cause a hang if rank to GPU mapping is heterogeneous. You can specify device_id in init_process_group()
CONFIG {'run': '